In [1]:
# in code below think of time as the rows of a df and columns as price path of df
# code below assumes zero for interest rates

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from Methods_xlWings import *

In [3]:
class MonteCarlo():

    def __init__(self):

        # ----------------------
        # Parameters
        # ----------------------
        self.initial_price = 10_000
        self.strikes = np.arange(2_000, 25_001, 1_000)
        
        self.rate = 0.00
        self.vol = 0.50

        self.yrs_to_expiry = 1.0
        self.days_in_year = 365
        
        self.paths = 100_000
        
        self.dt = self.yrs_to_expiry / self.days_in_year        
        self.steps = int(self.days_in_year * self.yrs_to_expiry)
        
        np.random.seed(43)
        self.Z_norm1 = np.random.normal(0, 1, size=(self.steps, self.paths))
        self.Z_norm2 = np.random.normal(0, 1, size=(self.steps, self.paths))

        self.Z_posneg1 = np.random.choice([-1, 1], size=(self.steps, self.paths))
        self.Z_posneg2 = np.random.choice([-1, 1], size=(self.steps, self.paths))

        self.S = np.zeros((self.steps+1, self.paths))
        self.S[0] = self.initial_price
        
        self.V = np.zeros((self.steps+1, self.paths))

    def sim_black_scholes(self):
        drift_term = self.rate * self.dt
        vol_sqrt_dt = self.vol * np.sqrt(self.dt)
        correction_term = -0.5 * vol_sqrt_dt**2
        for t in range(1, self.steps+1):         
            vol_term = vol_sqrt_dt * self.Z_norm1[t-1]
            self.S[t] = self.S[t-1] * np.exp(drift_term + vol_term + correction_term)

    def sim_local_vol(self, beta):
        drift_term = self.rate * self.dt
        for t in range(1, self.steps + 1): 
            vol_t = self.vol * (np.maximum(self.S[t-1], 1e-8) / self.initial_price) ** (-beta)
            vol_term = vol_t * np.sqrt(self.dt) * self.Z_norm1[t-1]
            correction_term = -0.5 * vol_t**2 * self.dt
            self.S[t] = self.S[t-1] * np.exp(drift_term + vol_term + correction_term)

    def sim_heston(self):
        pass

    def sim_merton_jump_diffusion(self):
        pass

    def sim_bates(self):
        pass

    def sim_cev(self):
        pass

    def sim_variance_gamma(self):
        pass                     

#    def Antithetic variates (easy, powerful):
    # for every path include its opposite

#    def Control variates (advanced, very classic)
    # adjust paths to get closed form known (forward price, black-scholes option price, etc.)

#    def Quasi–Monte Carlo (Sobol / Halton)

    def sim_option_prices(self, final_prices, strikes, discount_factor=1.0):    
        call_payoffs = np.maximum(final_prices[:, None] - strikes[None, :], 0)
        put_payoffs  = np.maximum(strikes[None, :] - final_prices[:, None], 0)
    
        call_prices = discount_factor * call_payoffs.mean(axis=0)
        put_prices  = discount_factor * put_payoffs.mean(axis=0)
        parity = strikes + call_prices - put_prices

        df = pd.DataFrame({'Strikes': strikes, 'Calls':call_prices, 'Puts':put_prices, 'Parity':parity})

        return df

    def plot_stuff(self, strikes, call_prices, put_prices):
        plt.figure(figsize=(12,6))
        plt.plot(strikes, call_prices, label='Call', color='blue', linestyle='--')
        plt.plot(strikes, put_prices, label='Put', color='red', linestyle='--')
        plt.xlabel("Strike")
        plt.ylabel("Option Price")
        plt.title("Option Prices")
        plt.legend()
        plt.grid(True)
        plt.show()


Monte Carlo Engine
│
├── Dynamics Models
│   ├── Black–Scholes
│   ├── Local Vol
│   ├── Heston
│   ├── Merton
│   └── Bates
│
└── Vol Surface Models
    ├── SABR (Hagan)
    ├── SVI / SSVI
    └── Market Surface
